# T08 — AR Model (AutoRegressive) — Book: CH05

**Methodology**: Marco Peixeiro, *Time Series Forecasting in Python*, Chapter 5.

### Book-mandated steps:
1. ADF stationarity test on health_index (level + first difference)
2. ACF and PACF plots to visually determine lag order p
3.  — select p by lowest AIC via SARIMAX(order=(p,0,0))
4. Fit best model → Ljung-Box residual test
5.  — walk-forward validation on representative engine
6. Full-dataset prediction → evaluate with RMSE + NASA score

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
# !pip install sklearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
import sklearn

# Book imports — exactly as CH05 uses them
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.models.classical import (
    build_pca_health_index, compute_failure_threshold,
    run_stationarity_report, plot_acf_pacf, smooth_series,
    optimize_AR, check_residuals,
    rolling_forecast_engine, predict_rul_ar,
    predict_dataset, simulate_test_from_train, RUL_CAP,
)
from src.evaluation.metrics import evaluate, evaluate_per_subset

PROC_DIR    = ROOT / "data" / "processed"
SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load data and build health_index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")

train, test = build_pca_health_index(train, test, SENSOR_COLS)

# Per-dataset_id thresholds — fixes global threshold bug
THRESHOLD_MAP = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print("Failure thresholds per dataset:", THRESHOLD_MAP)

## 2. Stationarity check — ADF test (CH03 methodology)

Book rule: run ADF at level + first difference. If level p > 0.05 and diff-1 p < 0.05 → d=1.
Here we check a stratified sample across ALL 4 FD subsets, not just 6 engines from FD001.

In [ ]:
# ADF on stratified sample: 5 engines per dataset_id × 4 subsets = 20 engines
stationarity_df = run_stationarity_report(train, n_engines_per_subset=5)

## 3. ACF and PACF plots (CH05 methodology)

Book rule: PACF cuts off at lag p → AR order.
Plot on smoothed health_index (smoothing is applied before fitting in production too).

In [ ]:
# Pick one representative engine from FD001 to show ACF/PACF
eid   = train[train.dataset_id == 1].engine_id.iloc[0]
raw   = train[train.engine_id == eid].sort_values("cycle").health_index.values
smth  = smooth_series(raw, window=5)

print(f"Engine {eid} | length: {len(smth)} cycles")
plot_acf_pacf(smth, lags=20, title=f"health_index — engine {eid} (FD001)")
print("Reading: PACF cuts off at lag p → candidate AR(p)")

## 4. Optimize AR order by AIC (CH05/CH06 pattern)

Book rule: run optimize function, sort by AIC ascending, pick lowest.

In [ ]:
# Use the same representative engine for order selection
# (AIC on a representative engine is more principled than RMSE on simulated val)
p_candidates = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

ar_aic_df = optimize_AR(smth, p_values=p_candidates)
print(ar_aic_df.to_string(index=False))

BEST_P = int(ar_aic_df.iloc[0]["p"])
print(f"
→ Best AR order: p = {BEST_P}  (lowest AIC)")

## 5. Fit best AR model and check residuals (Ljung-Box)

Book rule (CH06/CH07): always run Ljung-Box after fitting. All p-values > 0.05 = white-noise residuals = adequate model.

In [ ]:
model_fit = SARIMAX(smth, order=(BEST_P, 0, 0), simple_differencing=False).fit(disp=False)
print(model_fit.summary())

lb_result = check_residuals(model_fit.resid, model_name=f"AR({BEST_P})")

## 6. Rolling forecast on representative engine (CH05 pattern)

Book rule: walk-forward validation — refit at each window step, predict out-of-sample.

In [ ]:
TRAIN_LEN = int(len(smth) * 0.7)  # 70% history as initial train, 30% as held-out
WINDOW    = 1                       # one-step-ahead (most realistic per book CH05)

pred_ar = rolling_forecast_engine(
    series    = smth,
    train_len = TRAIN_LEN,
    order     = (BEST_P, 0, 0),
    window    = WINDOW,
)

actual_val = smth[TRAIN_LEN:TRAIN_LEN + len(pred_ar)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(len(smth)), smth, color="steelblue", label="observed health_index")
ax.plot(range(TRAIN_LEN, TRAIN_LEN + len(pred_ar)), pred_ar,
        color="darkorange", linestyle="--", label=f"AR({BEST_P}) rolling forecast")
ax.axvline(TRAIN_LEN, color="gray", linestyle=":", label="train/val split")
ax.axhline(THRESHOLD_MAP.get(1, 0), color="red", linestyle="--", label="failure threshold")
ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
ax.set_title(f"Rolling forecast — AR({BEST_P}) — engine {eid}")
ax.legend(); plt.tight_layout(); plt.show()

rmse_roll = float(np.sqrt(np.mean((actual_val - pred_ar)**2)))
print(f"Rolling forecast RMSE on health_index: {rmse_roll:.4f}")

## 7. Full test-set evaluation

In [ ]:
predict_fn = partial(predict_rul_ar, p=BEST_P)
y_true, y_pred, dids = predict_dataset(test, predict_fn, THRESHOLD_MAP)

print("
=== AR Model — Test Results ===")
test_metrics = evaluate_per_subset(y_true, y_pred, dids, model_name=f"AR({BEST_P})")
display(test_metrics)